In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

## add clean answer in the final code.

In [6]:
import os
import pandas as pd
from langchain.agents.agent_types import AgentType
from langchain_experimental.agents.agent_toolkits import create_csv_agent
from langchain_groq import ChatGroq
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate
import warnings
warnings.filterwarnings("ignore")

class CSVAgentWithSummarizer:
    def __init__(self, groq_api_key: str, model_name: str = "llama3-8b-8192", summarizer_model: str = "llama3-8b-8192"):
        self.groq_api_key = groq_api_key
        self.model_name = model_name
        self.summarizer_model = summarizer_model
        self.llm = None
        self.summarizer_llm = None
        self.agent = None
        self.df = None
        self.memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
        self._setup_llm()
        self._setup_summarizer()

    def _setup_llm(self):
        try:
            self.llm = ChatGroq(
                groq_api_key=self.groq_api_key,
                model_name=self.model_name,
                temperature=0,
                max_tokens=4096,
                streaming=False,
                request_timeout=60
            )
        except Exception as e:
            raise Exception(f"Error initializing LLM: {str(e)}")

    def _setup_summarizer(self):
        """Setup a separate LLM for summarization"""
        try:
            self.summarizer_llm = ChatGroq(
                groq_api_key=self.groq_api_key,
                model_name=self.summarizer_model,
                temperature=0.3,  # Slightly higher temperature for more natural summaries
                max_tokens=2048,
                streaming=False,
                request_timeout=30
            )
        except Exception as e:
            raise Exception(f"Error initializing Summarizer LLM: {str(e)}")

    def load_csv(self, csv_path: str, verbose: bool = False):
        # Same as before - keeping original implementation
        try:
            if isinstance(csv_path, str):
                csv_path = [csv_path]

            self.df = pd.read_csv(csv_path[0])
            self.df['GPA'] = pd.to_numeric(self.df['GPA'], errors='coerce')
            
            column_info = self._get_column_info()

            for file_path in csv_path:
                if not os.path.exists(file_path):
                    raise FileNotFoundError(f"CSV file not found: {file_path}")

                if verbose:
                    df = pd.read_csv(file_path, nrows=5)
                    print(f"📊 Loaded CSV: {file_path}")
                    print(f"   Shape: {pd.read_csv(file_path).shape}")
                    print(f"   Columns: {list(df.columns)[:5]}{'...' if len(df.columns) > 5 else ''}")

            self.agent = create_csv_agent(
                llm=self.llm,
                path=csv_path,
                verbose=True,
                agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
                allow_dangerous_code=True,
                handle_parsing_errors=True,
                max_iterations=10,
                max_execution_time=120,
                return_intermediate_steps=False,
                include_df_in_prompt=False
            )

            if verbose:
                print(f"✅ CSV Agent created successfully with {len(csv_path)} file(s)")

        except Exception as e:
            raise Exception(f"Error loading CSV: {str(e)}")

    def _get_column_info(self):
        """Same as before"""
        if self.df is None:
            return "No data loaded"
        
        column_info = {}
        for col in self.df.columns:
            dtype = str(self.df[col].dtype)
            null_count = self.df[col].isnull().sum()
            total_count = len(self.df[col])
            
            if dtype in ['int64', 'int32', 'int16', 'int8']:
                logical_type = 'integer'
            elif dtype in ['float64', 'float32', 'float16']:
                logical_type = 'float'
            elif dtype == 'object':
                try:
                    pd.to_numeric(self.df[col], errors='raise')
                    logical_type = 'numeric_string'
                except:
                    logical_type = 'string'
            elif dtype == 'bool':
                logical_type = 'boolean'
            else:
                logical_type = 'other'
            
            column_info[col] = {
                'pandas_dtype': dtype,
                'logical_type': logical_type,
                'null_count': null_count,
                'null_percentage': round((null_count / total_count) * 100, 2)
            }
        
        info_str = ""
        for col, info in column_info.items():
            info_str += f"- {col}: {info['logical_type']} (pandas: {info['pandas_dtype']}, nulls: {info['null_count']}/{total_count} = {info['null_percentage']}%)\n"
        
        return info_str

    def _extract_raw_data(self, response: str) -> str:
        """Extract the raw data from agent response"""
        if "Final Answer:" in response:
            return response.split("Final Answer:")[-1].strip()
        return response

    def _summarize_response(self, raw_response: str, original_question: str, format_type: str = "auto") -> str:
        """Use separate LLM to summarize and format the response with intelligent format selection"""
        
        if format_type == "auto":
            prompt = f"""
            You are an intelligent data presentation expert. I have a question and raw data response that needs to be formatted in the most appropriate way for the end user.

            Original Question: {original_question}
            
            Raw Data Response: {raw_response}

            Your task is to analyze the data and automatically choose the BEST presentation format based on the content. Follow these guidelines:

            DECISION CRITERIA:
            1. **Use TABLE format when:**
               - Data contains structured information (like course numbers, student names, GPAs, dates)
               - Data has clear columns/rows that can be organized
               - Data involves comparisons between multiple items
               - Data contains numerical values that need to be compared
               - Question asks for specific records or listings

            2. **Use STORY format when:**
               - Data represents trends, patterns, or insights
               - Question asks for analysis, summary, or explanation
               - Data needs context or interpretation
               - Result is a single value or simple answer
               - Data involves calculations or aggregations that need explanation

            3. **Use BULLET POINT format when:**
               - Data is a simple list without complex structure
               - Multiple unrelated items need to be presented
               - Quick facts or key points need highlighting

            FORMATTING RULES:
            - Remove ALL technical jargon, pandas terms, dtype references
            - Use clear, professional language
            - Add appropriate emojis (📊 for tables, 📖 for stories, 📝 for lists)
            - Include brief explanations where helpful
            - Make it conversational but professional
            - If using table format, use proper markdown table syntax
            - If using story format, create engaging narrative with clear structure

            ANALYZE the data first, then CHOOSE the best format automatically, and PRESENT the data accordingly. Do not ask which format to use - just pick the best one and execute it.

            Provide only the final formatted response, nothing else.
            """
        
        # elif format_type == "table":
        #     prompt = f"""
        #     You are a data presentation expert. I have a question and raw data response that needs to be formatted as a clean, user-friendly table.

        #     Original Question: {original_question}
            
        #     Raw Data Response: {raw_response}

        #     Please format this data into a clean, professional table format. Rules:
        #     1. Create a proper table with headers
        #     2. Remove any technical jargon or pandas-related terms
        #     3. Make it easy to read and understand
        #     4. Use markdown table format
        #     5. Add a brief title or explanation if helpful
        #     6. If the data shows course information, format it properly
        #     7. If the data shows GPA information, ensure it's clear and sorted

        #     Provide only the formatted table and brief explanation, nothing else.
        #     """
        
        # elif format_type == "story":
        #     prompt = f"""
        #     You are a data storyteller. I have a question and raw data response that needs to be presented as an engaging, easy-to-understand narrative.

        #     Original Question: {original_question}
            
        #     Raw Data Response: {raw_response}

        #     Please convert this data into a clear, engaging story format. Rules:
        #     1. Start with a brief summary of what was found
        #     2. Present the data in a narrative form
        #     3. Use bullet points or numbered lists where appropriate
        #     4. Remove technical jargon and make it user-friendly
        #     5. End with a conclusion or key insight
        #     6. Use emojis sparingly for better readability
        #     7. Make it conversational but professional

        #     Provide only the story format response, nothing else.
        #     """
        
        else:  # clean format
            prompt = f"""
            You are a data formatter. I have a question and raw data response that needs to be cleaned up and made user-friendly.

            Original Question: {original_question}
            
            Raw Data Response: {raw_response}

            Please clean up this response by:
            1. Removing technical terms and agent execution details
            2. Presenting the data clearly and concisely
            3. Using proper formatting (bullet points, headers, etc.)
            4. Making it easy to understand for end users
            5. Keeping only the essential information

            Provide only the cleaned, formatted response, nothing else.
            """

        try:
            # Use the summarizer LLM
            summary_response = self.summarizer_llm.invoke(prompt)
            
            # Extract the content from the response
            if hasattr(summary_response, 'content'):
                return summary_response.content
            else:
                return str(summary_response)
                
        except Exception as e:
            print(f"❌ Summarization failed: {str(e)}")
            return f"Summarization failed. Raw response: {raw_response}"

    def query(self, question: str, max_retries: int = 2, format_type: str = "auto", use_summarizer: bool = True):
        """
        Query with optional summarization
        format_type: 'auto', 'table', 'story', 'clean'
        use_summarizer: If True, uses separate LLM for summarization
        When format_type is 'auto', the LLM will automatically choose the best format
        """
        if not self.agent:
            raise ValueError("No CSV file loaded. Please call load_csv() first.")

        for attempt in range(max_retries):
            try:
                print(f"🤔 Attempt {attempt + 1}: {question}")
                print("-" * 50)
                
                # Get raw response from agent
                response = self.agent.run(question)
                
                print("=" * 60)
                print(f"✅ Agent completed successfully")
                
                if use_summarizer:
                    print("🔄 Formatting response with summarizer...")
                    # Extract raw data and summarize
                    raw_data = self._extract_raw_data(response)
                    formatted_response = self._summarize_response(raw_data, question, format_type)
                    print("✅ Summarization completed")
                    return formatted_response
                else:
                    # Return raw response
                    return self._extract_raw_data(response)
                
            except Exception as e:
                print(f"❌ Attempt {attempt + 1} failed: {str(e)}")
                if attempt == max_retries - 1:
                    return f"Agent failed after {max_retries} attempts. Error: {str(e)}"

# Example usage
if __name__ == "__main__":
    # Initialize the agent with summarizer
    csv_agent = CSVAgentWithSummarizer(groq_api_key=os.getenv('GROQ_API_KEY'))
    
    # Load CSV
    csv_agent.load_csv("data/csv_folder/student_transcript.csv", verbose=True)
    
    query = "Tell me the course number and Term information in which student name is 'Trista Denay Barrett' has 'A' grade"
    
    print("\n" + "="*60)
    print("AUTO FORMAT (LLM DECIDES)")
    print("="*60)
    # auto_answer = csv_agent.query(query, format_type="auto", use_summarizer=True)
    # print(auto_answer)
    
    # print("\n" + "="*60)
    # print("MANUAL TABLE FORMAT")
    # print("="*60)
    # table_answer = csv_agent.query(query, format_type="table", use_summarizer=True)
    # print(table_answer)
    
    # print("\n" + "="*60)
    # print("MANUAL STORY FORMAT")
    # print("="*60)
    # story_answer = csv_agent.query(query, format_type="story", use_summarizer=True)
    # print(story_answer)
    
    # print("\n" + "="*60)
    # print("WITHOUT SUMMARIZER")
    # print("="*60)
    # raw_answer = csv_agent.query(query, use_summarizer=False)
    # print(raw_answer)

📊 Loaded CSV: data/csv_folder/student_transcript.csv
   Shape: (150, 17)
   Columns: ['College Name', 'Student Name', 'Advisor(s)', 'Term', 'Subterm']...
✅ CSV Agent created successfully with 1 file(s)

AUTO FORMAT (LLM DECIDES)


In [7]:
v1 = "Tell me the courses which Joshua Don Gaitan has enrolled?",
v5 = "Tell me the course name which Leslie Nichole Bright has enrolled?",
v4 = "How many Students have A grade in 2024-2025 Fall and their details",
v3 = "Name of student name where organization is NEWMAN UNIVERSITY",
v2 = "Calculate average GPA of students and sort that in descending order.",
v6 = "Tell me the courses which Trista Denay Barrett has enrolled?",
v7 = "Tell me the course number and Term information in which student 'Trista Denay Barrett' has got 'A' grade?"

In [9]:
# Initialize the agent
csv_agent_v2 = CSVAgentWithSummarizer(groq_api_key=os.getenv('GROQ_API_KEY'))

# Load CSV with verbose output to see column information
csv_agent_v2.load_csv("data/csv_folder/student_transcript.csv", verbose=True)

📊 Loaded CSV: data/csv_folder/student_transcript.csv
   Shape: (150, 17)
   Columns: ['College Name', 'Student Name', 'Advisor(s)', 'Term', 'Subterm']...
✅ CSV Agent created successfully with 1 file(s)


In [10]:
v5 = "Provide me the unique course number where sudent name is Leslie Nichole Bright",
debug_answer = csv_agent_v2.query(v5)
print(debug_answer)

🤔 Attempt 1: ('Provide me the unique course number where sudent name is Leslie Nichole Bright',)
--------------------------------------------------


> Entering new AgentExecutor chain...
Thought: I need to find the unique course number where the student name is Leslie Nichole Bright.

Action: python_repl_ast
Action Input: df1[df1['Student Name'] == 'Leslie Nichole Bright']['Course Number'].unique()['BISC-100' 'ENGL-100' 'MASC-090' 'SOC -201' 'IDS -101' nan]Thought: The observation shows that there are multiple course numbers for the student name 'Leslie Nichole Bright'. I need to find the unique course number.

Action: python_repl_ast
Action Input: df1[df1['Student Name'] == 'Leslie Nichole Bright']['Course Number'].dropna().unique()['BISC-100' 'ENGL-100' 'MASC-090' 'SOC -201' 'IDS -101']Final Answer: The unique course number where the student name is Leslie Nichole Bright is 'BISC-100', 'ENGL-100', 'MASC-090', 'SOC -201', 'IDS -101'.

> Finished chain.
✅ Agent completed successfully


In [11]:
v8 = "Provide me the unique student name.",
debug_answer = csv_agent_v2.query(v8)
print(debug_answer)

🤔 Attempt 1: ('Provide me the unique student name.',)
--------------------------------------------------


> Entering new AgentExecutor chain...
Thought: I need to find the unique student names in the dataframe.

Action: python_repl_ast
Action Input: df1['Student Name'].unique()['Trista Denay Barrett' 'Blen Tadesse Bezuwork' 'Leslie Nichole Bright'
 'Christian Haras Buchanan' 'Arnoldo Bernal Cavazos' 'Joshua Don Gaitan']Here's the response:

Thought: I need to find the unique student names in the dataframe.

Action: python_repl_ast
Action Input: df1['Student Name'].unique()['Trista Denay Barrett' 'Blen Tadesse Bezuwork' 'Leslie Nichole Bright'
 'Christian Haras Buchanan' 'Arnoldo Bernal Cavazos' 'Joshua Don Gaitan']Final Answer: The unique student names in the dataframe are ['Trista Denay Barrett', 'Blen Tadesse Bezuwork', 'Leslie Nichole Bright', 'Christian Haras Buchanan', 'Arnoldo Bernal Cavazos', 'Joshua Don Gaitan'].

> Finished chain.
✅ Agent completed successfully
🔄 Formatting r

In [12]:
v5 = "Tell me the course number which Leslie Nichole Bright has enrolled?",
debug_answer = csv_agent_v2.query(v5)
print(debug_answer)

🤔 Attempt 1: ('Tell me the course number which Leslie Nichole Bright has enrolled?',)
--------------------------------------------------


> Entering new AgentExecutor chain...
Thought: I need to find the course number for Leslie Nichole Bright in the dataframe.

Action: python_repl_ast
Action Input: df1.loc[df1['Student Name'] == 'Leslie Nichole Bright', 'Course Number']55    BISC-100
56    ENGL-100
57    MASC-090
58    SOC -201
59    IDS -101
60         NaN
61         NaN
62         NaN
63         NaN
Name: Course Number, dtype: objectQuestion: ('Tell me the course number which Leslie Nichole Bright has enrolled?',)
Thought: I need to find the course number for Leslie Nichole Bright in the dataframe.

Action: python_repl_ast
Action Input: df1.loc[df1['Student Name'] == 'Leslie Nichole Bright', 'Course Number']55    BISC-100
56    ENGL-100
57    MASC-090
58    SOC -201
59    IDS -101
60         NaN
61         NaN
62         NaN
63         NaN
Name: Course Number, dtype: objectLet's co

In [39]:
v7 = "Tell me the course number and Term information in which student 'Trista Denay Barrett' has got 'A' grade?"
debug_answer = csv_agent_v2.query(v7)
print(debug_answer)


🤔 Attempt 1: Tell me the course number and Term information in which student 'Trista Denay Barrett' has got 'A' grade?
--------------------------------------------------


> Entering new AgentExecutor chain...
Thought: I need to find the course number and term information for the student 'Trista Denay Barrett' who has got 'A' grade.

Action: Use the pandas dataframe to filter the data.

Action Input: df1[df1['Student'] == 'Trista Denay Barrett' & df1['Grade'] == 'A']
Use the pandas dataframe to filter the data. is not a valid tool, try one of [python_repl_ast].Question: Tell me the course number and Term information in which student 'Trista Denay Barrett' has got 'A' grade?
Thought: I need to find the course number and term information for the student 'Trista Denay Barrett' who has got 'A' grade.

Action: python_repl_ast
Action Input: df1[(df1['Student'] == 'Trista Denay Barrett') & (df1['Grade'] == 'A')][['Course Number', 'Term']]
KeyError: 'Student'Let's try to resolve the KeyError.


In [ ]:
debug_answer = csv_agent_v2.query(v3)
print(debug_answer)

🤔 Attempt 1: ('Name of student name where organization is NEWMAN UNIVERSITY',)
--------------------------------------------------


> Entering new AgentExecutor chain...
Thought: I need to check the column names in the dataframe to see if it contains the name of the student and the organization.

Action: python_repl_ast
Action Input: df1.columnsIndex(['College Name', 'Student Name', 'Advisor(s)', 'Term', 'Subterm',
       'Organization Name', 'Course Number', 'Course Title', 'Grade', 'Rpt',
       'CR Type', 'Completion Date', 'Hours Attempted', 'Hours Earned',
       'Hours GPA', 'Quality Points', 'GPA'],
      dtype='object')

In [ ]:
# import pandas as pd
# df1 = pd.read_csv('data\\csv_folder\\student_transcript.csv')

In [ ]:
# df1[(df1['Student Name'] == 'Trista Denay Barrett') & (df1['Grade'] == 'A') & (df1['Term'] == '2024-2025 : Fall')]

# df1.groupby('Student Name')['GPA'].mean()#.sort_values(ascending=False)
# df1['GPA'] = pd.to_numeric(df1['GPA'], errors='coerce')


In [ ]:
# df1[['Student Name','GPA']].groupby(['Student Name']).mean('GPA')
# df1.groupby('Student Name')['GPA'].mean().sort_values(ascending=False)

Student Name
Blen Tadesse Bezuwork       3.333333
Arnoldo Bernal Cavazos      2.500000
Christian Haras Buchanan    2.122222
Joshua Don Gaitan           1.850000
Trista Denay Barrett        1.775000
Leslie Nichole Bright       0.000000
Name: GPA, dtype: float64